<a href="https://colab.research.google.com/github/aykahsay/Multilogual_transaltion_nlp/blob/main/notebooks/colab_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
# 📢 MASTER MULTILINGUAL PSA TRANSLATION STUDIO
## Fine-Tuning NMT Models for English ↔ Swahili ↔ Ekegusii (Gusii)

This master notebook combines both translation pipelines into a single, fully self-contained workflow:
1. **PART 1**: Meta NLLB-200 Fine-Tuning for English ➡️ Ekegusii
2. **PART 2**: MarianMT Fine-Tuning for English ➡️ Swahili
3. **PART 3**: Automatic Evaluation (SacreBLEU & chrF)
4. **PART 4**: Interactive Trilingual Inference & Batch Translation

**⚡ Important:** Enable GPU acceleration via **Runtime > Change runtime type > T4 GPU** before executing cells!

# ========================================================
# 🛠️ SETUP & DATA PREPARATION
# ========================================================

In [ ]:
# 1. Clone GitHub Repository & Change Directory
!git clone https://github.com/aykahsay/Multilogual_transaltion_nlp.git
%cd Multilogual_transaltion_nlp

# 2. Install Required NLP & ML Libraries
!pip install -q transformers datasets evaluate sacrebleu sentencepiece sacremoses torch accelerate pandas scikit-learn tqdm

# 3. Verify Hardware Acceleration (GPU)
import torch
print(f"PyTorch Version : {torch.__version__}")
print(f"GPU Available   : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Model       : {torch.cuda.get_device_name(0)}")

In [ ]:
# 4. Load Master Dataset and Filter Language Subsets
import os
import pandas as pd

master_path = "data/Master_PSA_Only.csv"
pending_marker = "N/A - Pending Fine-Tuned Model Inference"

df = pd.read_csv(master_path, dtype=str)
print(f"Loaded {len(df):,} total rows from {master_path}")

# Filter valid parallel pairs
has_swa = df['Kiswahili'].fillna('').str.strip().ne('') & (df['Kiswahili'] != pending_marker)
has_guz = df['Ekegusii'].fillna('').str.strip().ne('') & (df['Ekegusii'] != pending_marker)

en_sw_df = df[has_swa][['English', 'Kiswahili', 'Domain']].reset_index(drop=True)
en_guz_df = df[has_guz][['English', 'Ekegusii', 'Domain']].reset_index(drop=True)
trilingual_df = df[has_swa & has_guz][['English', 'Kiswahili', 'Ekegusii', 'Domain']].reset_index(drop=True)

print(f"✅ English - Swahili Parallel Pairs : {len(en_sw_df):,}")
print(f"✅ English - Ekegusii Parallel Pairs: {len(en_guz_df):,}")
print(f"✅ Complete Trilingual Triplets    : {len(trilingual_df):,}")

# ========================================================
# 🇰🇪 PART 1: FINE-TUNING META NLLB-200 (ENGLISH ➡️ EKEGUSII)
# ========================================================
Fine-tunes `facebook/nllb-200-distilled-600M` on low-resource Ekegusii parallel sentences (`eng_Latn` -> `guz_Latn`).

In [ ]:
import torch
from transformers import (
    AutoTokenizer, 
    AutoModelForSeq2SeqLM, 
    DataCollatorForSeq2Seq, 
    Seq2SeqTrainingArguments, 
    Seq2SeqTrainer
)
from datasets import Dataset

guz_model_checkpoint = "facebook/nllb-200-distilled-600M"
guz_output_dir = "models/nllb-en-guz"
os.makedirs(guz_output_dir, exist_ok=True)

print(f"Loading tokenizer & model: {guz_model_checkpoint}...")
guz_tokenizer = AutoTokenizer.from_pretrained(guz_model_checkpoint, src_lang="eng_Latn", tgt_lang="guz_Latn")
guz_model = AutoModelForSeq2SeqLM.from_pretrained(guz_model_checkpoint)

# Prepare Train / Validation Split (90% train, 10% val)
shuffled_guz = en_guz_df.sample(frac=1, random_state=42).reset_index(drop=True)
guz_split_idx = int(0.9 * len(shuffled_guz))

guz_train_data = Dataset.from_pandas(shuffled_guz.iloc[:guz_split_idx])
guz_val_data = Dataset.from_pandas(shuffled_guz.iloc[guz_split_idx:])

def preprocess_nllb(examples):
    inputs = [str(ex) for ex in examples["English"]]
    targets = [str(ex) for ex in examples["Ekegusii"]]
    model_inputs = guz_tokenizer(inputs, max_length=128, truncation=True)
    labels = guz_tokenizer(text_target=targets, max_length=128, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

print("Tokenizing Ekegusii datasets...")
guz_tokenized_train = guz_train_data.map(preprocess_nllb, batched=True, remove_columns=guz_train_data.column_names)
guz_tokenized_val = guz_val_data.map(preprocess_nllb, batched=True, remove_columns=guz_val_data.column_names)

guz_collator = DataCollatorForSeq2Seq(guz_tokenizer, model=guz_model)

guz_training_args = Seq2SeqTrainingArguments(
    output_dir=guz_output_dir,
    eval_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=3,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    save_strategy="epoch",
    report_to="none"
)

guz_trainer = Seq2SeqTrainer(
    model=guz_model,
    args=guz_training_args,
    train_dataset=guz_tokenized_train,
    eval_dataset=guz_tokenized_val,
    processing_class=guz_tokenizer,
    data_collator=guz_collator,
)

print("\n🔥 Starting NLLB-200 Ekegusii Fine-Tuning...")
guz_trainer.train()

# Save fine-tuned model
guz_trainer.save_model(guz_output_dir)
guz_tokenizer.save_pretrained(guz_output_dir)
print(f"✅ Ekegusii Model saved to {guz_output_dir}")

# ========================================================
# 🌍 PART 2: FINE-TUNING MARIANMT (ENGLISH ➡️ SWAHILI)
# ========================================================
Fine-tunes `Helsinki-NLP/opus-mt-en-sw` on English-Swahili parallel sentences.

In [ ]:
from transformers import MarianTokenizer, AutoModelForSeq2SeqLM

sw_model_checkpoint = "Helsinki-NLP/opus-mt-en-sw"
sw_output_dir = "models/psa-en-sw-finetuned"
os.makedirs(sw_output_dir, exist_ok=True)

print(f"Loading tokenizer & model: {sw_model_checkpoint}...")
sw_tokenizer = MarianTokenizer.from_pretrained(sw_model_checkpoint)
sw_model = AutoModelForSeq2SeqLM.from_pretrained(sw_model_checkpoint)

# Prepare Train / Validation Split (90% train, 10% val)
shuffled_sw = en_sw_df.sample(frac=1, random_state=42).reset_index(drop=True)
sw_split_idx = int(0.9 * len(shuffled_sw))

sw_train_data = Dataset.from_pandas(shuffled_sw.iloc[:sw_split_idx])
sw_val_data = Dataset.from_pandas(shuffled_sw.iloc[sw_split_idx:])

def preprocess_marian(examples):
    inputs = [str(ex) for ex in examples["English"]]
    targets = [str(ex) for ex in examples["Kiswahili"]]
    model_inputs = sw_tokenizer(inputs, text_target=targets, max_length=128, padding="max_length", truncation=True)
    model_inputs["labels"] = [
        [(l if l != sw_tokenizer.pad_token_id else -100) for l in label] for label in model_inputs["labels"]
    ]
    return model_inputs

print("Tokenizing Swahili datasets...")
sw_tokenized_train = sw_train_data.map(preprocess_marian, batched=True, remove_columns=sw_train_data.column_names)
sw_tokenized_val = sw_val_data.map(preprocess_marian, batched=True, remove_columns=sw_val_data.column_names)

sw_collator = DataCollatorForSeq2Seq(sw_tokenizer, model=sw_model)

sw_training_args = Seq2SeqTrainingArguments(
    output_dir=sw_output_dir,
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=3,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    save_strategy="epoch",
    report_to="none"
)

sw_trainer = Seq2SeqTrainer(
    model=sw_model,
    args=sw_training_args,
    train_dataset=sw_tokenized_train,
    eval_dataset=sw_tokenized_val,
    processing_class=sw_tokenizer,
    data_collator=sw_collator,
)

print("\n🔥 Starting Swahili MarianMT Fine-Tuning...")
sw_trainer.train()

# Save fine-tuned model
sw_trainer.save_model(sw_output_dir)
sw_tokenizer.save_pretrained(sw_output_dir)
print(f"✅ Swahili Model saved to {sw_output_dir}")

# ========================================================
# 📈 PART 3: AUTOMATIC EVALUATION (BLEU / SACREBLEU / CHRF)
# ========================================================
Computes quantitative evaluation metrics comparing model predictions against ground truth target sentences.

In [ ]:
import evaluate
from tqdm import tqdm

print("Loading evaluation metrics...")
sacrebleu_metric = evaluate.load("sacrebleu")
chrf_metric = evaluate.load("chrf")

# Evaluate 100 validation samples for Ekegusii
eval_sample = guz_val_data.select(range(min(100, len(guz_val_data))))
references = [[str(ex)] for ex in eval_sample["Ekegusii"]]
inputs = [str(ex) for ex in eval_sample["English"]]

predictions = []
print("Generating Ekegusii translation predictions...")
for text in tqdm(inputs):
    input_ids = guz_tokenizer(text, return_tensors="pt").input_ids.to(guz_model.device)
    outputs = guz_model.generate(input_ids, max_length=128)
    pred_text = guz_tokenizer.decode(outputs[0], skip_special_tokens=True)
    predictions.append(pred_text)

bleu_res = sacrebleu_metric.compute(predictions=predictions, references=references)
chrf_res = chrf_metric.compute(predictions=predictions, references=references)

print("\n========================================")
print("  AUTOMATIC EKEGUSII EVALUATION RESULTS")
print("========================================")
print(f"  • SacreBLEU Score : {bleu_res['score']:.2f}")
print(f"  • chrF Score      : {chrf_res['score']:.2f}")
print("========================================")

# ========================================================
# 🧪 PART 4: INTERACTIVE TRILINGUAL TRANSLATION STUDIO
# ========================================================
Interactive inference engine for single sentences and batch translations.

In [ ]:
def translate_psa(english_text):
    # 1. Swahili Translation
    sw_inputs = sw_tokenizer(english_text, return_tensors="pt").input_ids.to(sw_model.device)
    sw_outs = sw_model.generate(sw_inputs, max_length=128)
    sw_text = sw_tokenizer.decode(sw_outs[0], skip_special_tokens=True)
    
    # 2. Ekegusii Translation
    guz_inputs = guz_tokenizer(english_text, return_tensors="pt").input_ids.to(guz_model.device)
    guz_outs = guz_model.generate(guz_inputs, max_length=128)
    guz_text = guz_tokenizer.decode(guz_outs[0], skip_special_tokens=True)
    
    return {
        "English": english_text,
        "Kiswahili": sw_text,
        "Ekegusii": guz_text
    }

sample_input = "Wash your hands frequently with soap and running water to prevent the spread of diseases."
res = translate_psa(sample_input)

print("📢 TRILINGUAL TRANSLATION RESULT:")
print(f"  [English]  : {res['English']}")
print(f"  [Kiswahili]: {res['Kiswahili']}")
print(f"  [Ekegusii] : {res['Ekegusii']}")